# Orbital Mech Tutorials
Quick Jupyter code to go over what I can do for the simulations for certain questions

## Tutorial 1

### Question 4
Finding Local Sidereal Time

Steps:
Find the date and time format
If East or West, add 180 degrees if needed 
    $$JD = 367 * y - round(\frac{(7 * (y + round(\frac{m+9}{12}) ) }{ (4))} ) + round(\frac{275 m}{9}) + d + 1721013.5 + \frac{UT}{24}$$
    $$T0 =  \frac{JD - 2451545}{36525} $$
    $$\theta_{G0} = 100.4606184 + 36000.77004 T0 + 0.000387933 T0^2 - 2.583×10^{-8} T0^3$$

For theta_G0, keep it between the bound of $0 < \theta \le 360$

For west or east, either keep it negative or positive.
East is positive

In [24]:
import numpy as np
import pandas as pd


# Finding the local sidereal time
# Input parameters
opt = 3

if opt == 1:
    # Sweden
    long_1 = 18
    long_2 = 3
    Lambda = long_1 + long_2 / 60
    y, m, d = 2008, 1, 1
    UT = 12

elif opt == 2:
    # California
    long_1 = 118
    long_2 = 15
    Lambda = - (long_1 + long_2 / 60)
    y, m, d = 2005, 7, 4
    UT = 20

else:
    # Me
    long_1 = 57.007
    long_2 = 0
    Lambda = - (long_1 + long_2 / 60)
    y, m, d = 2026, 4, 14
    UT = 13

## J0
JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24

## T0
T0 = (JD - 2451545) / 36525
print(T0)

## theta G0
theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
print(theta_G0)

## theta G
theta_G =  theta_G0 + 360.98564724 * UT/24
while theta_G > 360:
    theta_G -= 360

print(theta_G)

## Local Sideral Time
LocalSiderealTime = theta_G + Lambda
print(LocalSiderealTime)

0.2628348163358388
9562.716426610941
38.25031886594115
-18.756681134058844


### Question 6
Orbital elements determination from radial and velocity vectors

Solution, follow a multitude of steps for you to get all orbital elements
1. Get magnitude of $|\underline{r}|$ and $|\underline{v}|$

2. Get $\underline{h}$ and magnitude of $|\underline h|$
$$ \underline h = \underline r \times \underline v $$

3. Find SMA
- from vis viva equation
$$\varepsilon = \frac{v^2}{2} - \frac{\mu}{r}$$
- Making the subsitution of $\varepsilon = \frac{-\mu}{2a}$
$$\frac{v^2}{2} = \frac{\mu}{r} - \frac{\mu}{a}$$
- Finally rearranging to get SMA
$$a = \frac{\mu}{\frac{2\mu}{r} - v^2}$$

4. Find ECC and magnitude $|\underline e |$
$$\underline e = \frac{1}{\mu} [\underline v \times \underline h - \mu \frac{\underline r}{r}]$$

5. Find the inclination angle
$$i = acos\left(\frac{h_z}{h}\right)$$

6. Find the nodal vector and the magnitude
$$\underline N = i_z \times \underline h$$

7. Find the RAAN
- **if N_y > 0** :
    $$\Omega = \arccos(\frac{N_x}{N})$$
- else
    $$\Omega = 360 - \arccos(\frac{N_x}{N})$$


8. Find the AOP
- **if e_z > 0**:
    $$\omega = \arccos(\frac{\underline N \cdot \underline e}{|\underline N||\underline e|})$$
- else:
    $$\omega = 360 - \arccos(\frac{\underline N \cdot \underline e}{|\underline N||\underline e|})$$


9. Find the True Anomaly

- **if $\underline r \cdot \underline v > 0$** :
    $$\theta = \arccos(\frac{\underline r \cdot \underline e}{|\underline r||\underline e|})$$
- else:
    $$\theta = 360 - \arccos(\frac{\underline r \cdot \underline e}{|\underline r||\underline e|})$$



In [1]:
import numpy as np
from math import cos, acos, pi

# constants
mu = 398600

# User parameters
opt = 5
if opt == 1:
    # q1 
    r = np.array([0.9412, -5.9851, 8.3585])
    v = np.array([0.4365, 0.127, 0.5661])
elif opt == 2:
    r = np.array([-6.0955, -11.8452, 11.0082])
    v = np.array([-0.0153, -0.7556, -0.044])
elif opt == 3:
    r = np.array([-1.7371, 0.0256, 0.1635])
    v = np.array([-0.0153, -0.7556, -0.0440])
elif opt == 4:
    r = np.array([-6200, -3560, 3100])
    v = np.array([-3.4, 6.3, 2.5])
elif opt == 5:
    r = np.array([
        -10527 + 12/99 * -1559,
        3039 + 12/99 * 450,
        5264 + 12/99 * 779
    ])
    v = np.array([
        -4.959,
        -1.432,
        -2.480
    ])

# Magnitude of r and v
R = np.linalg.norm(r)
V = np.linalg.norm(v)
print("r mag = " , R)
print("v mag = " , V)

# angular momentum
h = np.cross(r, v)
H = np.linalg.norm(h)
print("h = " , h)
print("h mag = " , H)


# semi major axis
a = mu / (2 * mu / R - (V)**2)
print("a = " , a)

# Eccentricity
e = 1/mu * (np.cross(v, h) - mu * r / R)
emag = np.linalg.norm(e)
print(np.cross(v, h))
print("e = ", e)
print("e mag = ", emag)

# inclination
i = acos(h[2] / H) * 180 / pi
print("i = ",i)

# Nodal vector
N = np.cross([0, 0, 1], h)
Nmag = np.linalg.norm(N)
print("N = ", N)
print("N magnitude = " , Nmag)

# RAAN
if N[1] > 0:
    print("bug")
    Omega = acos(N[0] / Nmag) * 180 / pi
else:
    Omega = 360 - acos(N[0] / Nmag) * 180 / pi
print("Omega = " , Omega)

# AOP
if e[2] > 0:
    omega = acos(np.dot(N, e) / (Nmag * emag)) * 180 / pi
else:
    omega = 360 - acos(np.dot(N, e) / (Nmag * emag)) * 180 / pi
print("omega = ", omega)

# theta
if np.dot(r, v) > 0:
    theta = acos(np.dot(r, e) / (R * emag)) * 180 / pi
else:
    theta = 360 -  acos(np.dot(r, e) / (R * emag)) * 180 / pi
print("theta =", theta)

print("Final results:")
print("a = ", a)
print(e)
print(i)
print(Omega)
print(omega)
print(theta)

r mag =  12373.954112874004
v mag =  5.726491508768698
h =  [ 1.27078788e+00 -5.31480307e+04  3.06861605e+04]
h mag =  61370.624996992534
a =  12600.788117152468
[-175749.69791103  152169.5184407   263562.90384424]
e =  [0.42509268 0.13175536 0.22818096]
e mag =  0.5001297935061827
i =  59.99908580711499
N =  [ 5.31480307e+04  1.27078788e+00 -0.00000000e+00]
N magnitude =  53148.03068185915
bug
Omega =  0.0013699619577650036
omega =  31.7914823622343
theta = 118.20608326488792
Final results:
a =  12600.788117152468
[0.42509268 0.13175536 0.22818096]
59.99908580711499
0.0013699619577650036
31.7914823622343
118.20608326488792


### Question 7
Orbital elements into radial and velocity vectors

Solution, follow the multitude of steps for you to find the final answer.
The more or less roundabout is to find the value of $|\underline v_p|$ and $|\underline r_p|$ through eccentricity equation.

1. Find the true anomaly from M.
$$E = M + e \sin E$$
$$\tan \frac{\theta}{2} = \tan \frac{E}{2} \sqrt{\frac{1 + e}{1 - e}}$$

2. Find the angular momentum
$$\frac{h^2}{2} = a (1 - e^2)$$

3. Find the radius and velocity vectors in the **perifocal** frame
$$r = \frac{h^2}{\mu (1 + e \cos \theta)} (\cos \theta i_e + \sin \theta i_p)$$
$$v = \frac{\mu}{h} (-e \sin \theta i_e + (e + \cos \theta) i_e)$$

4. Use the transformation matrix from perifocal frame to ECI frame, check the data sheet

Tutorial solution

In [51]:
import numpy as np
from math import cos, radians, sin, atan, degrees, tan, sqrt

a = 4.58
e = 0.8
i = 87
Omega = 93
omega = 202
M = 90

# Normalised question
mu = 1

# Find true anomaly
E = 1
for j in range(1, 200):
    E = radians(M) + e * sin(E)
print("E = ", E, "rads (or ", degrees(E), "deg)")

theta = 2 * atan( tan(E/2) * sqrt((1 + e) / (1 - e)))
print("theta = ", theta, "rads (or ", degrees(theta), "deg)")

# Find the angular momentum
h = sqrt(mu * a * ( 1 - e**2))
print("h = ", h)

# Find r and v in perifocal frame
r = np.array(
    [h**2 / mu * 1 / (1 + e * cos(theta)) * cos(theta), # i_e vector
    h**2 / mu * 1 / (1 + e * cos(theta)) * sin(theta), # i_p vector
    0]                                                 # i_h vector
)
print("r = ", r)

v = np.array(
    [mu / h * ( -np.sin(theta)) , # i_e vector
    mu / h * (e + np.cos(theta)), # i_p vector
    0]                                                 # i_h vector
)
print("v = ", v)

# Apply perifocal to ECI transformation
cO = np.cos(np.radians(Omega))
sO = np.sin(np.radians(Omega))
co = np.cos(np.radians(omega))
so = np.sin(np.radians(omega))
ci = np.cos(np.radians(i))
si = np.sin(np.radians(i))

perifocal_to_inertial_frame = np.array([
    [
        cO*co - sO*so*ci,
        -cO*so - sO*co*ci,
        sO*si
    ],
    [
        sO*co + cO*so*ci,
        -sO*so + cO*co*ci,
        -cO*si
    ],
    [
        so*si,
        co*si,
        ci
    ]
])

print("Transformation matrix:\n ", perifocal_to_inertial_frame)

r = perifocal_to_inertial_frame @ r
v = perifocal_to_inertial_frame @ v
print("\n\nFinal vectors:\n")
print("r: ", r, "\n")
print("v: ", v)

E =  2.2119306096084457 rads (or  126.73428850636327 deg)
theta =  2.81033528305589 rads (or  161.0203507358061 deg)
h =  1.2840560735419615
r =  [-6.40332026  2.20229626  0.        ]
v =  [-0.25328512 -0.11341728  0.        ]
Transformation matrix:
  [[ 0.06810358  0.02885316  0.99726095]
 [-0.92488711  0.37663281  0.05226423]
 [-0.37409321 -0.92591318  0.05233596]]


Final vectors:

r:  [-0.37254583  6.75180543  0.35630348] 

v:  [-0.02052207  0.19154347  0.1997668 ]


Calculator (used for mock exam)

In [8]:
import numpy as np
from math import radians

a = 26610.21
e = 0.747
theta = 24.31

# Non-normalised question
mu = 398600

# Find the eccentric anomaly
E = 2 * np.arctan(np.sqrt((1 - e) / (1 + e)) * np.tan(radians(theta)/2))
print("E = ", E)

# Find the mean anomaly
M = E - e * np.sin(E)
print("M = ", M)

# Find the time taken

## Periapsis
periapsis_t = M * np.sqrt(a**3 / mu)
## Apoapsis
apoapsis_t = (np.pi - M) * np.sqrt(a**3 / mu)

print(f"Time taken from periapsis: {periapsis_t}s or {periapsis_t / 3600} hrs")
print(f"Time taken from apoapsis: {apoapsis_t}s or {apoapsis_t / 3600} hrs")

E =  0.1635651860122654
M =  0.04192606985663318
Time taken from periapsis: 288.2623741991408s or 0.08007288172198357 hrs
Time taken from apoapsis: 21311.734004834132s or 5.9199261124539255 hrs


### Question 10
Time propagation

Solution
1. Find the angular momentum
$$|\underline h| = a (1 - e^2)$$
2. Find the mean anomaly
3. Add the delta time variation
4. Find the new true anomaly
5. Apply equations to find the perifocal frame velocities and radi
6. Apply rotation transformation matrix to get vectors in coordinates

In [43]:
import numpy as np
from math import sqrt, radians, degrees, pi

# Normalised coefficients
mu = 1

# Perturbation time
deltaT = 113

# Orbital elements
opt = 1
if opt == 1:
    a = 15
    e = 0.216
    i = 121
    Omega = 33
    omega = 275
    theta = 217
elif opt == 2:
    a = 4.58
    e = 0.8
    i = 87
    Omega = 93
    omega = 202
    theta = 161.02

# Find the angular momentum
h = sqrt(mu * a * (1 - e**2))
print("h = ", h)

# Find eccentricity anomaly
tanE2 = sqrt((1 - e) / (1 + e)) * np.tan(radians(theta)/2)
E = 2 * np.atan(tanE2)
print("E = " , E)

# Mean anomaly
M = E - e * np.sin(E)
print("M = ", M)

# Apply perturbation
newM = M + sqrt(mu / (a**3)) * deltaT
print("perturbed M = ", newM)

# Find new true anomaly
newE = 1
for j in range(0, 200):
    newE = newM - e * np.sin(newE)
newtanTheta2 = sqrt((1 + e) / (1 - e)) * np.atan(newE / 2)
newTheta = 2 * tan(newtanTheta2)
print("Perturbed eccentric anomaly:" , newE)
print("Perturbed true anomaly:" , newTheta)

# if  newTheta < 0 or newTheta > 2*pi:
#     newTheta += 2*pi
#     print("Altered true anomaly:" , newTheta)

# Find the new perifocal frame radial and velocity components
r = np.array([
    h**2 / mu * np.cos(newTheta),
    h**2 / mu * np.sin(newTheta),
    0
])

v = np.array([
    mu / h * (-e * np.sin(newTheta)),
    mu / h * (e + np.cos(newTheta)),
    0
])
print("Perifocal frame: ")
print("r = ", r)
print("v = ", v)

# Transform into inertial frame
cO = np.cos(np.radians(Omega))
sO = np.sin(np.radians(Omega))
co = np.cos(np.radians(omega))
so = np.sin(np.radians(omega))
ci = np.cos(np.radians(i))
si = np.sin(np.radians(i))

perifocal_to_inertial_frame = np.array([
    [
        cO*co - sO*so*ci,
        -cO*so - sO*co*ci,
        sO*si
    ],
    [
        sO*co + cO*so*ci,
        -sO*so + cO*co*ci,
        -cO*si
    ],
    [
        so*si,
        co*si,
        ci
    ]
])

print("Transformation matrix:\n ", perifocal_to_inertial_frame)

r = perifocal_to_inertial_frame @ r
v = perifocal_to_inertial_frame @ v
print("\n\nFinal vectors:\n")
print("r: ", r, "\n")
print("v: ", v)

h =  3.7815552356140456
E =  -2.35194524315448
M =  -2.1985625502476265
perturbed M =  -0.2534642474856794
Perturbed eccentric anomaly: -0.20870955836536992
Perturbed true anomaly: -0.2604468415598436
Perifocal frame: 
r =  [13.81788721 -3.6824678   0.        ]
v =  [0.01470894 0.31264253 0.        ]
Transformation matrix:
  [[-0.20634746  0.85992722  0.46684677]
 [ 0.477772    0.50491983 -0.71888099]
 [-0.85390552  0.07470705 -0.51503807]]


Final vectors:

r:  [ -6.0179402    4.74244864 -12.07427648] 

v:  [0.26581467 0.16488694 0.01079656]


### Lagrange's Coefficients Example

Example I would like to go through to check I understand the problem

1. Identify what the magnitude or $|\underline r|$ and $|\underline v|$
2. Find the $|\underline h|$, and $\theta $.
3. Convert to $M$ and add time propagations
4. Finalise with the by plugging in what $f, g, \dot f, \dot g$


In [ ]:
# Solutions
import numpy as np

# Constats
mu = 398600

# Input for the values of r and v
r0 = np.array([
    13140,
    -294,
    20126
])
v0 = np.array([
    2.658,
    3.567,
    -0.505
])

# Find the mangitudes
R0 = np.linalg.norm(r0)
V0 = np.linalg.norm(v0)
print("r mag = ", R0)
print("v mag = ", V0)

# Find the magnitude of h
h = R0 * V0
print("h = ", h)

# Find the SMA
a = (- mu) / (V0**2 - 2 * mu / R0)
print("a = ", a)
# Find the eccentricity
e = np.sqrt(1 - h**2/(mu * a))
print("e = ", e)

# Find the value of theta
costheta = (h**2 / (R0 * mu) - 1) * 1/e
theta = np.arccos(costheta)
print("Theta = ", theta)

# Convert to M
tanE2 = np.sqrt((1 - e)/(1 + e)) * np.tan(theta / 2)
E = 2 * np.arctan(tanE2)
print("E = ", E)
M = E - e * np.sin(E)

# Add time perturbation
deltaT = 4 * 3600
newM = M + np.sqrt(mu / a**3) * deltaT
print("New M = ", newM)


# Convert back to theta
newE = 1
for j in range(0, 200):
    newE = newM + e * np.sin(newE)
tantheta2 = np.sqrt((1 + e)/(1 - e)) * np.tan(newE / 2)
newtheta = 2 * np.arctan(tantheta2)
print("New E = ", newE)
print("New theta = ", newtheta)

# Find delta theta
deltatheta = newtheta - theta
print("Change in theta = ", deltatheta)

# Find back to velocity and radial positions using the formula of f and g
newR = h ** 2 / (mu * (1 + e * np.cos(newtheta)))
f = 1 - mu * newR / (h**2) * (1 - np.cos(deltatheta))
g = newR * R / h * np.sin(deltatheta)

fdot = mu / h * (1 - np.cos(deltatheta)) / np.sin(deltatheta) * (mu / (h**2) * (1 - np.cos(deltatheta) - 1 / R - 1/newR))
gdot = 1 - mu * R / h *(1 - np.cos(deltatheta))
print("f = ", f)
print("g = ", g)
print()
print("fdot = ", fdot)
print("gdot = ", gdot)

newr = f * r + g * v
newv = fdot * r + gdot * v

print("Final values")
print("new r = ", newr)
print("new v = ", newv)


r mag =  24037.5105200185
v mag =  4.476994304217953
h =  107615.79768570195
a =  30377.962334684722
e =  0.20871879900341006
Theta =  0.0
E =  0.0
New M =  1.717090479810947
New E =  1.913660905241581
New theta =  2.1050245799189304
Change in theta =  2.1050245799189304
f =  -0.688636496349244
g =  6249.665456584227

fdot =  0.00033734437427562606
gdot =  -134365.47504317976
Final values
new r =  [  7562.92722157  22495.01581356 -17015.5791811 ]
new v =  [-357138.99995969 -479281.74865827   67861.35428968]


## Tutorial 2

### Question 1 and 2
Gibb's method, determination of velocity and orbital elements

*Main purpose of using Gibb's method is for orbital elements determination and finding the velocity given 3 positions*

Steps to find the solution
1. Find the magnitudes of the 3 radial vectors $|\underline r_1|$, $|\underline r_2|$ & $|\underline r_3|$

2. Check if the 3 vectors are coplanar, a satisfiable check is that
$$\frac{\underline r_1}{r1} \cdot (\underline r_2 \times \underline r_3) \approx 0$$
- This check is necessary because we need to assess the severity of perturbations or sensor sensitivity for the results of the data

3. Calculate the auxiliary vectors
$$\mathbf{N} = \underline r_1 (\underline r_2 \times \underline r_3) + \underline r_2 (\underline r_3 \times \underline r_1) + \underline r_3 (\underline r_1 \times \underline r_2)$$
$$\mathbf{S} = \underline r_1 \times \underline r_2 + \underline r_2 \times \underline r_3 + \underline r_3 \times \underline r_1$$
$$\mathbf{D} = \underline r_1 (\underline r_2 - \underline r_3) + \underline r_2 (\underline r_3 - \underline r_1) + \underline r_3 (\underline r_1 - \underline r_2)$$

4. Apply the formula below
$$\underline v = \sqrt{\frac{\mu}{\mathbf N \mathbf D}} \left(\frac{\mathbf D \times \underline r}{r} + \mathbf S \right)$$

5. Find the orbital elements if need be

In [48]:
import numpy as np
from math import pi

mu = 398600
opt = 2

if opt == 1:
    r1 = np.array([
        5887,
        -3520,
        -1204,
    ])
    r2 = np.array([
        5572,
        -3457,
        -2376
    ])
    r3 = np.array([
        5088,
        -3289,
        -3480,
    ])
elif opt == 2:
    r1 = np.array([
        -6150,
        -669,
        -3851,
    ])
    r2 = np.array([
        -5353,
        -2187,
        -4759
    ])
    r3 = np.array([
        -4377,
        -3848,
        -5696,
    ])


# magnitudes
R1 = np.linalg.norm(r1)
R2 = np.linalg.norm(r2)
R3 = np.linalg.norm(r3)

print("r1 mag:" , R1)
print("r2 mag:" , R2)
print("r3 mag:" , R3)

# Check for plane between r1, r2, r3
# Criteria:
# r1 / r1_mag cdot (r2 cross r3) == 0
check = np.dot(r1, np.cross(r2, r3)) / R1
print(check)
# Well the check shows its non-planar
# But oh well.

## Checking cross product terms
print("r2 x r3 = ", np.cross(r2, r3))
print("r3 x r1 = ", np.cross(r3, r1))
print("r1 x r2 = ", np.cross(r1, r2))

# Auxilary vectors
## N
N = R1 * np.cross(r2, r3) + R2 * np.cross(r3, r1) + R3 * np.cross(r1, r2)
## S
S = r1 * (R2 - R3) + r2 * (R3 - R1) + r3 * (R1 - R2)
## D
D = np.cross(r1, r2) + np.cross(r2, r3) + np.cross(r3, r1)

print("N = ", N)
print("S = ", S)
print("D = ", D)

# Finding what v is
## Prelim cal of D cross r2
print("D x r2 = ", np.cross(D, r2))

## Our case specifically asked for v2
v2 = np.sqrt(mu / (np.linalg.norm(N) * np.linalg.norm(D))) * (np.cross(D, r2) / R2 + S)
print("v2 = ", v2)

r1 mag: 7286.992658154666
r2 mag: 7489.035919262238
r3 mag: 8149.211556954452
1863.2528721990402
r2 x r3 =  [-5855480 -9660545 11025845]
r3 x r1 =  [ 11008024  18174573 -20736987]
r1 x r2 =  [-5238366 -8653447  9868893]
N =  [-2.91790538e+09 -4.80506077e+09  5.46890795e+09]
S =  [ 328965.7603997  -666552.7613171  -410124.94336673]
D =  [ -85822 -139419  157751]
D x r2 =  [ 1008496458 -1252868001  -558617193]
v2 =  [  6.93188252 -12.46713971  -7.24716883]


In [49]:
from math import degrees, radians

# Orbital elements
print("r2 = ", r2)
print("v2 = ", v2)

V2 = np.linalg.norm(v2)
print("v2 mag = ", V2)

# h
h = np.cross(r2, v2)
h_mag = np.linalg.norm(h)
print("h = ", h)
print("h mag = ", h_mag)
print(R2 * V2)

# a
a = mu / (2 * mu / R2 - V2**2)
print("a = ", a)

# e
e = 1 / mu * (np.cross(v2, h) - mu * r2 / R2)
e_mag = np.linalg.norm(e)
print("e = ", e)
print("e mag = ", e_mag)

# i
i = np.arccos(h[2] / h_mag)
print("i = ", degrees(i))

# Omega
N = np.cross(np.array([0, 0, 1]), h)
N_mag = np.linalg.norm(N)
if N[1] > 0:
    Omega = np.arccos(N[0] / N_mag)
else:
    Omega = 2*pi - np.arccos(N[0] / N_mag)

print("Omega = ", degrees(Omega))

# omega
if e[2] > 0:
    omega = np.arccos(np.dot(N, e) / (N_mag * e_mag))
else:
    omega = 2*pi - np.arccos(np.dot(N, e) / (N_mag * e_mag))
print("omega = ", degrees(omega))

# theta
if np.dot(r2, v2) > 0:
    theta = np.arccos(np.dot(e, r2) / (e_mag * R2))
else:
    theta = 2*pi - np.arccos(np.dot(e, r2) / (e_mag * R2))
print("theta = ", degrees(theta))


r2 =  [-5353 -2187 -4759]
v2 =  [  6.93188252 -12.46713971  -7.24716883]
v2 mag =  16.00006324356585
h =  [-43481.55965122 -71782.9236356   81896.62593067]
h mag =  117262.48971519245
119825.04834153214
a =  -2665.274516104512
e =  [-3.15185407 -0.34163989 -1.97287054]
e mag =  3.734048752486278
i =  45.70088020870474
Omega =  328.7951672658
omega =  227.5799605008609
theta =  15.028696125954733


## Tutorial 3

### Question 2

#### b, *Solving for Cowell using numerical methods*


derived state space for Cowell's method


In [96]:
import numpy as np
from matplotlib import pyplot as plt

# Definitions
## Creating the Keplerian orbit
def f(state):
    r = state[:2]
    p = r * 10**(-2)
    v = state[2:]
    a = p - mu * r / np.linalg.norm(r)**3
    return np.hstack((v, a))

def K2(state):
    f0 = f(state)
    f1 = f(state + h * f0)

    return state + h/2 * (f0 + f1)




# Initial parameters
h = 0.814 # time-step
mu = 1 # constant

r = np.array([
    1,
    0,
])
v = np.array([
    0,
    1.2124,
])
state = np.hstack((r, v))
print(state)



# Applying Numerical Method
plt.figure()
for i in range(0, 1):
    state = K2(state)

print(state)

[1.     0.     0.     1.2124]
[ 0.67201498  0.9868936  -0.54561308  1.07158698]


<Figure size 640x480 with 0 Axes>


#### 2c, Encke's method

New state space


**Theory**

First, begin with splitting the orbit into Keplerian and Perturbed states

Keplerian:
$$
\ddot r + \frac{-\mu}{r^3} \underline r = 0
$$

Non-Keplerian:
$$
\ddot r + \frac{-\mu}{r^3} \underline r = \underline a_d
$$

Partition the position and velocity into Keplerian and pertubed states
$$
\underline r = \underline r_0 + \underline \delta \\
\underline v = \underline v_0 + \underline \nu
$$

Going through the lecture notes, you will arrive:
$$
\frac{d^2 \delta}{dt^2} + \frac{\mu}{r_0^3} \delta = \underline a_d + \frac{\mu}{r_0^3} \left(1 - \frac{r_0^3}{r^3}\right) \underline r
$$

We can simplify further into the $\underline \phi$ function using:
$$
\left(1 - \frac{r_0^3}{r^3}\right) = \phi(q) = -q \frac{3 + 3q + q^2}{1 + (1 + q)^{3/2}}
$$
And for the definition of $q$:
$$
q = \frac{\underline \delta \cdot (\underline \delta - 2 \underline r)}{\underline r \cdot \underline r}
$$

Thus leading to the final equation we need to use for perturbations:
$$
\frac{d^2 \delta}{dt^2} + \frac{\mu}{r_0^3} \delta = \underline a_d + \frac{\mu}{r_0^3} \phi(q) (\underline r_0 + \underline \delta)
\tag{1}
$$

**Implementation**
```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
flowchart TD
    A[Initialise <br/> delta r = 0$] --> B[Propagate Kepler]
    B --> C[Compute q function]
    C --> D[Find the acceleration <br/> of delta ddot r]
    D --> E[Integrate ODE step to find delta r and delta r dot]
    E -- Next time step --> A
```

In [ ]:
import numpy as np

# Definitions
## Creating the Keplerian orbit
def f(state, statedelta):
    
    r = state[:2]
    v = state[2:]
    
    p = r * 10**(-2)
    a = p - mu * r / np.linalg.norm(r)**3




    # Final perturbations
    return np.hstack((v, a))

def K2(state, statedelta):
    f0 = f(state, statedelta)
    f1 = f(state + h * f0, statedelta + h * f0)

    return state + h/2 * (f0 + f1)

def phi(state, statedelta):
    r = state[:2]
    deltar = statedelta[:2]

    q = float((np.dot(deltar, (deltar - 2*r))) / (np.dot(r, r)))
    
    phi = -q * (3 + 3*q + q**2) / (1 + (1 + q)**(3/2))
    return phi


# Initial parameters
h = 0.814 # time-step
mu = 1 # constant

## Keplerian
r0 = np.array([
    1,
    0,
])
v0 = np.array([
    0,
    1.2124,
])

## Perturbations
deltar = np.array([
    0,
    0
])
deltav = np.array([
    0,
    0
])

## Complete vectors
r = r0 + deltar
v = v0 + deltav

state0 = np.hstack((r0, v0))
statedelta = np.hstack((deltar, deltav))
state = np.hstack((r, v))



for i in range(0, 2):
    ## Keplerian
    state = K2(state)
    print(state)
    ## Encke
    
phi(state, statedelta)



[ 0.67201498  0.9868936  -0.54561308  1.07158698]
[ 0.09930919  1.67034288 -0.71675605  0.73204047]
0.0


-0.0

In [22]:
import numpy as np

mu = 1.0   # gravitational parameter
h  = 0.814 # time step

def perturbation(r):
    """Return perturbing acceleration f_p at position r."""
    return 1e-2 * r  # placeholder — replace with real force

def encke_accel(state_ref, state_delta):
    """
    Compute δr̈ using Encke's method.
    state_ref   : [r*, v*] — reference Keplerian state (4,)
    state_delta : [δr, δv] — deviation from reference    (4,)
    """
    r_ref   = state_ref[:2]
    dr      = state_delta[:2]
    r_true  = r_ref + dr

    r_ref_norm2 = np.dot(r_ref, r_ref)

    # Battin q-function (correct sign)
    q   = np.dot(dr, 2*r_ref + dr) / r_ref_norm2
    fq  = -q * (3 + 3*q + q**2) / (1 + (1 + q)**1.5)

    r_ref_norm3 = r_ref_norm2**1.5
    dv  = state_delta[2:]
    ddr = -(mu / r_ref_norm3) * (dr - fq * r_true) + perturbation(r_true)

    return np.hstack((dv, ddr))

def keplerian_accel(state_ref):
    """Two-body acceleration for the reference orbit."""
    r = state_ref[:2]
    v = state_ref[2:]
    a = -mu * r / np.linalg.norm(r)**3
    return np.hstack((v, a))

def rk2_step(state, deriv_fn):
    """Heun's method (RK2) for a single time step."""
    f0 = deriv_fn(state)
    f1 = deriv_fn(state + h * f0)
    return state + h/2 * (f0 + f1)

# --- Initial conditions ---
r0 = np.array([1.0, 0.0])
v0 = np.array([0.0, 1.2124])

state_ref   = np.hstack((r0, v0))          # reference orbit
state_delta = np.zeros(4)                   # deviation starts at zero

# --- Propagation loop ---
n_steps = 100
for i in range(n_steps):
    state_ref   = rk2_step(state_ref,   keplerian_accel)
    state_delta = rk2_step(state_delta, lambda sd: encke_accel(state_ref, sd))

    # Rectify if deviation grows too large
    if np.linalg.norm(state_delta[:2]) / np.linalg.norm(state_ref[:2]) > 1e-3:
        state_ref   = state_ref + state_delta
        state_delta = np.zeros(4)
        print(f"  Rectified at step {i}")

    r_true = state_ref[:2] + state_delta[:2]
    print(f"Step {i:3d} | r = {r_true}")

  Rectified at step 0
Step   0 | r = [0.6709174  0.99016316]
  Rectified at step 1
Step   1 | r = [0.09500009 1.67952946]
  Rectified at step 2
Step   2 | r = [-0.50007909  2.17158101]
  Rectified at step 3
Step   3 | r = [-1.07255978  2.55476049]
  Rectified at step 4
Step   4 | r = [-1.62134252  2.87781334]
  Rectified at step 5
Step   5 | r = [-2.15310231  3.16840353]
  Rectified at step 6
Step   6 | r = [-2.67555629  3.44364364]
  Rectified at step 7
Step   7 | r = [-3.19606619  3.71505633]
  Rectified at step 8
Step   8 | r = [-3.72143139  3.99100371]
  Rectified at step 9
Step   9 | r = [-4.25795871  4.27796771]
  Rectified at step 10
Step  10 | r = [-4.81158527  4.58127537]
  Rectified at step 11
Step  11 | r = [-5.38799869  4.90553729]
  Rectified at step 12
Step  12 | r = [-5.99274304  5.25492871]
  Rectified at step 13
Step  13 | r = [-6.63131061  5.63337969]
  Rectified at step 14
Step  14 | r = [-7.30922232  6.04471056]
  Rectified at step 15
Step  15 | r = [-8.03209978  6.

## Tutorial 4

### Question 2

Uses the Equation
$$
-\Omega^2 x = -\frac{\mu_1}{r_1^3} (x + \mu_2^* r_{12}) - \frac{\mu_2}{r_2^3} (x - \mu_1^* r_{12})
$$

Find values for:
- $\Omega$
- $r_1$
- $r_2$
- $\mu_1^*$
- $\mu_2^*$

Solve the 3 degree polynomial from there. Reason why its coded out is because its a horrible solve to go through so sympy is used (MATLAB equivalent of the symbolic math toolbox)

In [44]:
import numpy as np
from math import pi
import sympy as sp

x = sp.symbols("x")
r12 = 149.6e6

opt = 1

if opt == 1:
    mu_E = 398600
    mu_S = 1.32712e11
elif opt == 2:
    mu_E = 797200
    mu_S = 2.65424e11


# Finding Omega
Omega = 2 * pi / (365.25 * 3600 * 24)
print("Omega = ", Omega)

# Centre of gravity
x_cg = mu_E * r12 / (mu_E + mu_S)
print("Centre of gravity = ", x_cg)

# Finding r1 and r2 
r1 = sp.Abs(x_cg + x)
r2 = sp.Abs(r12 - x_cg - x)

print(x)
print(r1)
print(r2)


# Find mu*
mu_Est = mu_E / (mu_E + mu_S)
mu_Sst = mu_S / (mu_E + mu_S)

print("mu_E* = ", mu_Est)
print("mu_S* = ", mu_Sst)

# Final equation
eq = (Omega**2 * x 
      - mu_S / (r1**3) * (x + mu_Est * r12)
      - mu_E / (r2**3) * (x - mu_Sst * r12))
# eq = Omega**2 * x - mu_S / r1**2 + mu_E / r2**2
# print(eq)

print("\n\nSolutions to the code: ")
guess_L1 = 149.6e6 - 1.5e6
x_L1 =float(sp.nsolve(eq, x, guess_L1))
print(f"x_L1 = {x_L1:.4e}")

guess_L2 = 149.6e6 + 1.5e6 
x_L2 = float(sp.nsolve(eq, x, guess_L2))
print(f"x_L2 = {x_L2:.4e}")

guess_L3 = -149.6e6
x_L3 = float(sp.nsolve(eq, x, guess_L3))
print(f"x_L3 = {x_L3:.4e}")


Omega =  1.991021277657232e-07
Centre of gravity =  449.3216958554768
x
Abs(x + 449.321695855477)
Abs(x - 149599550.678304)
mu_E* =  3.0034872717612084e-06
mu_S* =  0.9999969965127282


Solutions to the code: 
x_L1 = 1.4811e+08
x_L2 = 1.5110e+08
x_L3 = -1.4960e+08


### Question 3

#### a, Going from orbital elements to cartesian

*May be a potential slip up in the answers from Amato, please double check*

In [1]:
import numpy as np
from math import cos, radians, sin, sqrt

a = 26610
e = 0.747
i = 0
Omega = 0
omega = 0
theta = radians(24.31)

mu = 398600

# Find the angular momentum
h = sqrt(mu * a * ( 1 - e**2))
print("h = ", h)

# Find r and v in perifocal frame
r = np.array(
    [h**2 / mu * 1 / (1 + e * cos(theta)) * cos(theta), # i_e vector
    h**2 / mu * 1 / (1 + e * cos(theta)) * sin(theta), # i_p vector
    0]                                                 # i_h vector
)
print("r = ", r)

v = np.array(
    [mu / h * (-np.sin(theta)) , # i_e vector
    mu / h * (e + np.cos(theta)), # i_p vector
    0]                                                 # i_h vector
)
print("v = ", v)

# Apply perifocal to ECI transformation
cO = np.cos(np.radians(Omega))
sO = np.sin(np.radians(Omega))
co = np.cos(np.radians(omega))
so = np.sin(np.radians(omega))
ci = np.cos(np.radians(i))
si = np.sin(np.radians(i))

perifocal_to_inertial_frame = np.array([
    [
        cO*co - sO*so*ci,
        -cO*so - sO*co*ci,
        sO*si
    ],
    [
        sO*co + cO*so*ci,
        -sO*so + cO*co*ci,
        -cO*si
    ],
    [
        so*si,
        co*si,
        ci
    ]
])

print("Transformation matrix:\n ", perifocal_to_inertial_frame)

r = perifocal_to_inertial_frame @ r
v = perifocal_to_inertial_frame @ v
print("\n\nFinal vectors:\n")
print("r: ", r, "\n")
print("v: ", v)

h =  68469.60107438921
r =  [6377.16663448 2880.74118523    0.        ]
v =  [-2.39658218  9.65407862  0.        ]
Transformation matrix:
  [[ 1. -0.  0.]
 [ 0.  1. -0.]
 [ 0.  0.  1.]]


Final vectors:

r:  [6377.16663448 2880.74118523    0.        ] 

v:  [-2.39658218  9.65407862  0.        ]
